In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Wikipedia NLI Fact-Checking Pipeline

Computes two new signals per sample:
- `wiki_score` — DeBERTa NLI between caption and Wikipedia summaries of entities
- `article_score` — DeBERTa NLI between caption and first 2 sentences of article (fixes broken DeBERTa)

Uses `caption_entities_rel` from metadata — Wikipedia-normalized entity names already provided.
No spaCy or NER needed.

Output: `wiki_nli_scores.csv` with columns [id, wiki_score, article_score, entity_count, wiki_coverage]
Resumes automatically if interrupted.

In [8]:
import os
import json
import numpy as np
import pandas as pd
import torch
import wikipediaapi
from transformers import pipeline
from tqdm import tqdm
import warnings, time
warnings.filterwarnings('ignore')

# ── Paths ──
PROJECT_ROOT  = str(_cfg.ROOT)
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
VAL_META_PATH = os.path.join(DATASET_ROOT, 'metadata', 'val.json')
VAL_ANN_PATH  = os.path.join(DATASET_ROOT, 'merged_balanced', 'val.json')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
WIKI_CACHE    = os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv')
WIKI_PAGE_CACHE = os.path.join(PROJECT_ROOT, 'wiki_page_cache.json')  # avoids re-fetching same pages

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"cuda" if device == 0 else "cpu"}')

# ── Load val sample IDs ──
id_df = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
val_ids = set(id_df['id'].values)
print(f'Val samples: {len(val_ids)}')

# ── Load metadata ──
with open(VAL_META_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

# Filter to val samples only
metadata = {k: v for k, v in metadata.items() if k in val_ids}
print(f'Metadata entries for val: {len(metadata)}')

# ── Load article text paths ──
# Check if articles are pre-loaded in metadata or need to be read from files
sample = metadata[list(metadata.keys())[0]]
print(f'\nMetadata fields: {list(sample.keys())}')
print(f'article_path sample: {sample.get("article_path", "NOT FOUND")}')
print(f'full_article_path  : {sample.get("full_article_path", "NOT FOUND")}')

Device: cuda
Val samples: 3232
Metadata entries for val: 3232

Metadata fields: ['id', 'caption', 'image_path', 'article_path', 'full_article_path', 'caption_entities_spacy', 'caption_entities_rel', 'image_has_person', 'topic', 'source', 'timestamp', 'title', 'title_entities_spacy']
article_path sample: visual_news/origin/washington_post/articles/129699.txt
full_article_path  : visual_news/articles/washingtonpost/washingtonpost/www.washingtonpost.com:blogs:style-blog:wp:2015:06:11:cmt-awards-really-love-carrie-underwood-and-10-other-lessons-from-the-always-strange-show:.json


In [9]:
# ── Load article text ──
# Try to read article text from full_article_path

def load_article_text(meta):
    """Load first 2 sentences of article text for a sample."""
    # Try full_article_path first
    for path_key in ['full_article_path', 'article_path']:
        rel_path = meta.get(path_key, '')
        if not rel_path:
            continue
        # Try both with and without dataset root prefix
        candidates = [
            os.path.join(DATASET_ROOT, rel_path),
            os.path.join(PROJECT_ROOT, rel_path),
            rel_path,
        ]
        for full_path in candidates:
            if os.path.exists(full_path):
                try:
                    with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                        text = f.read().strip()
                    # Return first 2 sentences (split on . or \n)
                    sentences = [s.strip() for s in text.replace('\n', '. ').split('.') 
                                 if len(s.strip()) > 20]
                    return '. '.join(sentences[:2])
                except:
                    pass
    return ''

# Test on a few samples
print('Testing article loading...')
loaded = 0
for sid, meta in list(metadata.items())[:20]:
    text = load_article_text(meta)
    if text:
        loaded += 1
        if loaded == 1:
            print(f'  Sample article text: {text[:200]}')

print(f'  Article load success: {loaded}/20')
if loaded == 0:
    print('  WARNING: Could not load any articles — article_score will be skipped')
    print('  Check article_path values in metadata vs actual file locations')
    # Show the raw path to debug
    sample = list(metadata.values())[0]
    print(f'  Raw article_path: {sample.get("article_path", "")}') 
    print(f'  Raw full_article_path: {sample.get("full_article_path", "")}')

Testing article loading...
  Article load success: 0/20
  Check article_path values in metadata vs actual file locations
  Raw article_path: visual_news/origin/washington_post/articles/129699.txt
  Raw full_article_path: visual_news/articles/washingtonpost/washingtonpost/www.washingtonpost.com:blogs:style-blog:wp:2015:06:11:cmt-awards-really-love-carrie-underwood-and-10-other-lessons-from-the-always-strange-show:.json


In [3]:
# ── Load DeBERTa NLI model ──
# Uses cross-encoder/nli-deberta-v3-small — faster than large, still accurate
# If you have the large model cached already, change to cross-encoder/nli-deberta-v3-large

print('Loading DeBERTa NLI model...')
nli = pipeline(
    'zero-shot-classification',
    model='cross-encoder/nli-deberta-v3-small',
    device=device,
)
print('NLI model loaded.')

def get_nli_score(premise, hypothesis, max_premise_len=400):
    """
    Returns entailment probability of hypothesis given premise.
    Truncates premise to max_premise_len chars for speed.
    Returns 0.33 (neutral prior) on failure.
    """
    if not premise or not hypothesis:
        return 0.33
    try:
        premise = premise[:max_premise_len]
        result  = nli(premise, candidate_labels=[hypothesis], 
                      hypothesis_template='{}', multi_label=False)
        return float(result['scores'][0])
    except:
        return 0.33

# Quick sanity check
score = get_nli_score(
    'Barack Obama was the 44th President of the United States.',
    'Barack Obama served as US President'
)
print(f'Sanity check (should be high ~0.7+): {score:.4f}')

score2 = get_nli_score(
    'Barack Obama was the 44th President of the United States.',
    'Donald Trump served as US President'
)
print(f'Sanity check (should be low ~0.1-0.3): {score2:.4f}')

Loading DeBERTa NLI model...


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 8153.54it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded.
Sanity check (should be high ~0.7+): 0.9999
Sanity check (should be low ~0.1-0.3): 0.0000


In [4]:
# ── Wikipedia fetcher with local cache ──
# Caches pages to disk — avoids re-fetching same entity across samples

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='PicsCanLie-Thesis/1.0 (research project)'
)

# Load existing page cache
if os.path.exists(WIKI_PAGE_CACHE):
    with open(WIKI_PAGE_CACHE, 'r', encoding='utf-8') as f:
        page_cache = json.load(f)
    print(f'Loaded wiki page cache: {len(page_cache)} entries')
else:
    page_cache = {}
    print('Starting fresh wiki page cache')

def get_wiki_summary(entity_name, max_chars=500):
    """
    Fetch Wikipedia summary for entity_name.
    entity_name should be Wikipedia-normalized (e.g. 'Barack_Obama').
    Returns summary string or '' if not found.
    Caches results to avoid duplicate fetches.
    """
    if entity_name in page_cache:
        return page_cache[entity_name]

    try:
        # Wikipedia-normalized names use underscores — convert to spaces for lookup
        lookup_name = entity_name.replace('_', ' ')
        page = wiki.page(lookup_name)
        if page.exists():
            summary = page.summary[:max_chars]
        else:
            summary = ''
    except:
        summary = ''

    page_cache[entity_name] = summary
    return summary

def save_page_cache():
    with open(WIKI_PAGE_CACHE, 'w', encoding='utf-8') as f:
        json.dump(page_cache, f, ensure_ascii=False)

# Test
summary = get_wiki_summary('Barack_Obama')
print(f'Test fetch Barack_Obama: {summary[:150]}...')

Starting fresh wiki page cache
Test fetch Barack_Obama: Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A mem...


In [5]:
# ── Simplified scoring loop — wiki_score only, no article files needed ──

SAVE_EVERY       = 50
WIKI_CACHE_EVERY = 200
MIN_CONF         = 0.75
MAX_ENTITIES     = 3

if os.path.exists(WIKI_CACHE):
    wiki_df       = pd.read_csv(WIKI_CACHE)
    wiki_df['id'] = wiki_df['id'].astype(str)
    already_done  = set(wiki_df['id'].values)
    print(f'Resuming — {len(already_done)} already scored')
else:
    wiki_df      = pd.DataFrame()
    already_done = set()
    print('Starting fresh')

remaining   = [sid for sid in id_df['id'].values if sid not in already_done]
new_results = []
no_entities = 0
no_wiki     = 0
errors      = 0

print(f'Samples to score: {len(remaining)}')
print('Saves every 50 samples — safe to interrupt and resume.')

for i, sample_id in enumerate(tqdm(remaining, desc='Wiki NLI scoring')):
    try:
        meta    = metadata.get(sample_id, {})
        caption = meta.get('caption', '')

        # ── Entity extraction from caption_entities_rel ──
        raw_entities  = meta.get('caption_entities_rel', [])
        valid_entities = [
            e for e in raw_entities
            if isinstance(e, list) and len(e) >= 4 and float(e[3]) >= MIN_CONF
        ]
        valid_entities = sorted(valid_entities, key=lambda e: float(e[3]), reverse=True)
        valid_entities = valid_entities[:MAX_ENTITIES]

        # ── Also use title as extra context for NLI premise ──
        title = meta.get('title', '')

        # ── Wikipedia NLI ──
        wiki_scores = []
        for entity in valid_entities:
            wiki_name    = entity[0]  # e.g. 'Barack_Obama'
            wiki_summary = get_wiki_summary(wiki_name)
            if wiki_summary:
                # Combine wiki summary with title for richer premise
                premise = wiki_summary
                if title:
                    premise = f"{wiki_summary} {title}"
                score = get_nli_score(premise, caption)
                wiki_scores.append(score)

        if not valid_entities:
            no_entities += 1
        elif not wiki_scores:
            no_wiki += 1

        new_results.append({
            'id'            : sample_id,
            'wiki_score'    : float(np.mean(wiki_scores))   if wiki_scores else 0.33,
            'wiki_score_min': float(np.min(wiki_scores))    if wiki_scores else 0.33,
            'wiki_score_max': float(np.max(wiki_scores))    if wiki_scores else 0.33,
            'entity_count'  : len(valid_entities),
            'wiki_coverage' : len(wiki_scores),
        })

    except Exception as e:
        errors += 1
        print(f'  ERROR on {sample_id}: {type(e).__name__}: {e}')
        new_results.append({
            'id': sample_id, 'wiki_score': 0.33, 'wiki_score_min': 0.33,
            'wiki_score_max': 0.33, 'entity_count': 0, 'wiki_coverage': 0,
        })

    # Save scores
    if len(new_results) % SAVE_EVERY == 0:
        partial  = pd.DataFrame(new_results)
        combined = pd.concat([wiki_df, partial], ignore_index=True) if not wiki_df.empty else partial
        combined.to_csv(WIKI_CACHE, index=False)

    # Save wiki page cache
    if (i + 1) % WIKI_CACHE_EVERY == 0:
        save_page_cache()
        print(f'  [Wiki cache: {len(page_cache)} pages at sample {i+1}]')

# Final saves
final_wiki = pd.DataFrame(new_results)
if not wiki_df.empty:
    final_wiki = pd.concat([wiki_df, final_wiki], ignore_index=True)
final_wiki.to_csv(WIKI_CACHE, index=False)
save_page_cache()

print(f'\nDone.')
print(f'  Scored       : {len(final_wiki)}')
print(f'  No entities  : {no_entities}')
print(f'  No wiki page : {no_wiki}')
print(f'  Errors       : {errors}')
print(f'  Pages cached : {len(page_cache)}')

Starting fresh
Samples to score: 5000
Saves every 50 samples — safe to interrupt and resume.


Wiki NLI scoring:   4%|▍         | 198/5000 [05:55<1:08:22,  1.17it/s]

  [Wiki cache: 293 pages at sample 200]


Wiki NLI scoring:   8%|▊         | 400/5000 [11:08<1:32:32,  1.21s/it]

  [Wiki cache: 546 pages at sample 400]


...

  [Wiki cache: 3143 pages at sample 4600]


Wiki NLI scoring:  96%|█████████▌| 4803/5000 [54:17<00:19, 10.22it/s]

  [Wiki cache: 3192 pages at sample 4800]


Wiki NLI scoring: 100%|██████████| 5000/5000 [54:57<00:00,  1.52it/s]

  [Wiki cache: 3245 pages at sample 5000]

Done.
  Scored       : 5000
  No entities  : 386
  No wiki page : 0
  Errors       : 0
  Pages cached : 3245


In [6]:
# ── Diagnostic — check if new signals are discriminative ──
# Run this after scoring is done

import pandas as pd
import numpy as np

wiki_df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)

id_df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'models', 'clip_finetuned_v2', 'val_features', 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)

merged = wiki_df.merge(id_df[['id', 'label']], on='id', how='inner')

print('=' * 60)
print('SIGNAL DISCRIMINABILITY (real vs fake)')
print('=' * 60)
for col in ['wiki_score', 'wiki_score_min', 'wiki_score_max', 'article_score']:
    real = merged[col][merged['label'] == 0]
    fake = merged[col][merged['label'] == 1]
    diff = fake.mean() - real.mean()
    print(f'\n  {col}:')
    print(f'    real: mean={real.mean():.4f}  std={real.std():.4f}')
    print(f'    fake: mean={fake.mean():.4f}  std={fake.std():.4f}')
    print(f'    diff: {diff:+.4f}  {"<-- useful" if abs(diff) > 0.01 else "<-- weak"}')

print(f'\n  Coverage:')
print(f'    Samples with entities    : {(merged["entity_count"] > 0).sum()}/{len(merged)}')
print(f'    Samples with wiki pages  : {(merged["wiki_coverage"] > 0).sum()}/{len(merged)}')
print(f'    Avg entities per sample  : {merged["entity_count"].mean():.2f}')
print(f'    Avg wiki pages per sample: {merged["wiki_coverage"].mean():.2f}')

SIGNAL DISCRIMINABILITY (real vs fake)

  wiki_score:
    real: mean=0.1535  std=0.1555
    fake: mean=0.1513  std=0.1522
    diff: -0.0022  <-- weak

  wiki_score_min:
    real: mean=0.0970  std=0.1509
    fake: mean=0.0944  std=0.1458
    diff: -0.0026  <-- weak

  wiki_score_max:
    real: mean=0.2180  std=0.2033
    fake: mean=0.2158  std=0.2013
    diff: -0.0022  <-- weak


KeyError: 'article_score'

In [11]:
import os, json
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT  = str(_cfg.ROOT)
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

# ── Load base signals ──
clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

# ── Load evidence scores ──
ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}

s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

# ── Load wiki scores ──
wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}

w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score',     0.33) for i in id_df['id']])
w2 = np.array([wiki_lookup.get(i, {}).get('wiki_score_min', 0.33) for i in id_df['id']])
w3 = np.array([wiki_lookup.get(i, {}).get('wiki_score_max', 0.33) for i in id_df['id']])
w4 = np.array([wiki_lookup.get(i, {}).get('entity_count',   0)    for i in id_df['id']])
w5 = np.array([wiki_lookup.get(i, {}).get('wiki_coverage',  0)    for i in id_df['id']])

print('All signals loaded.')
print(f'Samples: {len(id_df)} | Labels: real={int((labels==0).sum())} fake={int((labels==1).sum())}')

# ── Build full feature matrix ──
X_full = np.stack([clip_probs, clip_sims, deb_scores,
                   s2, s3, s4, s5, s6,
                   w1, w2, w3, w4, w5], axis=1)
print(f'Feature matrix: {X_full.shape}')

# ── Train XGBoost ──
X_train, X_val, y_train, y_val = train_test_split(
    X_full, labels, test_size=0.2, random_state=42, stratify=labels)

xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     eval_metric='logloss', early_stopping_rounds=30,
                     random_state=42)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

preds = xgb.predict(X_val)
print(f'\nAccuracy : {accuracy_score(y_val, preds)*100:.2f}%')
print(f'F1       : {f1_score(y_val, preds):.4f}')

print('\nFeature importances:')
names = ['clip_prob','clip_sim','deberta','s2','s3','s4','s5','s6',
         'wiki_mean','wiki_min','wiki_max','entity_count','wiki_coverage']
for n, imp in zip(names, xgb.feature_importances_):
    print(f'  {n:<15}: {imp:.4f}')

All signals loaded.
Samples: 5000 | Labels: real=2500 fake=2500
Feature matrix: (5000, 13)
[0]	validation_0-logloss:0.66290
[50]	validation_0-logloss:0.30282
[100]	validation_0-logloss:0.28581
[144]	validation_0-logloss:0.28781

Accuracy : 87.20%
F1       : 0.8733

Feature importances:
  clip_prob      : 0.4637
  clip_sim       : 0.1600
  deberta        : 0.0207
  s2             : 0.0288
  s3             : 0.0337
  s4             : 0.0401
  s5             : 0.0842
  s6             : 0.0401
  wiki_mean      : 0.0242
  wiki_min       : 0.0252
  wiki_max       : 0.0241
  entity_count   : 0.0213
  wiki_coverage  : 0.0340


In [ ]:
# ── Progress check — run anytime without interrupting main loop ──

import pandas as pd
df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'wiki_nli_scores.csv'))
scored = len(df)
total  = 5000
with_wiki    = (df['wiki_coverage'] > 0).sum()
with_article = (df['article_score'] != 0.33).sum()
print(f'Progress       : {scored}/{total} ({scored/total*100:.1f}%)')
print(f'With wiki data : {with_wiki}/{scored} ({with_wiki/max(scored,1)*100:.1f}%)')
print(f'With article   : {with_article}/{scored} ({with_article/max(scored,1)*100:.1f}%)')